# 04 Model Training & Evaluation

Training four classifiers with SMOTE, comparing them, and evaluating the best one in depth.  
This experimental work led directly to `src/train.py` and `src/score.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

print('All imports OK')

## Rebuild preprocessed data

Self-contained - replicates the preprocessing and feature engineering steps inline.

In [ ]:
df = pd.read_csv('../data/raw/telco_churn.csv')

# Clean
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Engineer
add_ons = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
           'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)
df['num_services']      = sum((df[s] == 'Yes').astype(int) for s in add_ons)
df['high_value']        = (df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)).astype(int)
df['tenure_group']      = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                                  labels=['0-1yr','1-2yr','2-4yr','4-6yr'])

# Encode
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']
le = LabelEncoder()
for col in X.select_dtypes(include=['object', 'category']).columns:
    X[col] = le.fit_transform(X[col].astype(str))

feature_names = X.columns.tolist()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_sc  = scaler.transform(imputer.transform(X_test))

print(f'Train: {X_train_sc.shape}   Test: {X_test_sc.shape}')
print(f'Class balance — train: {y_train.mean()*100:.1f}% churn')

## Why SMOTE?

Class imbalance (27% churn) means a naive model predicts "no churn" for everything and still gets 73% accuracy. SMOTE creates synthetic minority-class samples so models learn the churn signal properly.

**Critical**: SMOTE is applied only to the training set, after the split. Applying it before would let synthetic test-like rows leak into training.

In [ ]:
print('Before SMOTE:')
print(y_train.value_counts())

X_res, y_res = SMOTE(random_state=42).fit_resample(X_train_sc, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_res).value_counts())

## Train four classifiers

In [ ]:
model_zoo = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.5, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
    'XGBoost'            : XGBClassifier(n_estimators=200, max_depth=4,
                                          eval_metric='logloss', random_state=42, verbosity=0),
}

results = {}
for name, model in model_zoo.items():
    model.fit(X_res, y_res)
    proba = model.predict_proba(X_test_sc)[:, 1]
    preds = model.predict(X_test_sc)
    results[name] = {
        'model'        : model,
        'proba'        : proba,
        'preds'        : preds,
        'roc_auc'      : roc_auc_score(y_test, proba),
        'avg_precision': average_precision_score(y_test, proba),
        'report'       : classification_report(y_test, preds, output_dict=True),
        'cm'           : confusion_matrix(y_test, preds),
    }
    print(f"{name:<25}  ROC-AUC: {results[name]['roc_auc']:.4f}")

## Model comparison table

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Model'        : name,
        'ROC-AUC'      : round(r['roc_auc'], 4),
        'Avg Precision': round(r['avg_precision'], 4),
        'Precision (1)': round(r['report']['1']['precision'], 4),
        'Recall (1)'   : round(r['report']['1']['recall'], 4),
        'F1 (1)'       : round(r['report']['1']['f1-score'], 4),
    })

comp = pd.DataFrame(rows).set_index('Model')
comp

In [ ]:
best_name = max(results, key=lambda k: results[k]['roc_auc'])
best      = results[best_name]
print(f'Best model: {best_name}  (ROC-AUC {best["roc_auc"]:.4f})')

## ROC & Precision-Recall curves — all models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, r in results.items():
    RocCurveDisplay.from_predictions(
        y_test, r['proba'], name=name, ax=axes[0]
    )
    PrecisionRecallDisplay.from_predictions(
        y_test, r['proba'], name=name, ax=axes[1]
    )

axes[0].set_title('ROC Curves')
axes[1].set_title('Precision-Recall Curves')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Best model — confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(best['cm'], display_labels=['Retained', 'Churned']).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = best['cm'].ravel()
print(f'Churners caught : {tp} / {tp+fn}  ({tp/(tp+fn)*100:.1f}% recall)')
print(f'False alarms    : {fp}  ({fp/(fp+tn)*100:.1f}% of retained flagged)')

## Feature importance

In [ ]:
model = best['model']

if hasattr(model, 'feature_importances_'):
    scores = model.feature_importances_
else:
    scores = np.abs(model.coef_[0])   # Logistic Regression

fi = pd.DataFrame({'feature': feature_names, 'importance': scores})
fi = fi.sort_values('importance', ascending=False).reset_index(drop=True)

top12 = fi.head(12)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top12['feature'][::-1], top12['importance'][::-1], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title(f'Top 12 Features — {best_name}')
plt.tight_layout()
plt.show()

fi.head(12)

## Threshold tuning

Default 0.5 threshold may not be optimal under class imbalance. Explore the precision-recall tradeoff.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.2, 0.8, 0.05)
rows = []
for t in thresholds:
    preds_t = (best['proba'] >= t).astype(int)
    rows.append({
        'threshold': round(t, 2),
        'precision': precision_score(y_test, preds_t, zero_division=0),
        'recall'   : recall_score(y_test, preds_t),
        'f1'       : f1_score(y_test, preds_t),
    })

thr_df = pd.DataFrame(rows).set_index('threshold')

fig, ax = plt.subplots(figsize=(8, 4))
thr_df.plot(ax=ax)
ax.axvline(0.5, color='black', linestyle='--', linewidth=0.8, label='default 0.5')
ax.set_title('Precision / Recall / F1 vs Threshold')
ax.set_xlabel('Threshold')
ax.legend()
plt.tight_layout()
plt.show()

best_f1_row = thr_df.loc[thr_df['f1'].idxmax()]
print(f'Best F1 at threshold {best_f1_row.name:.2f}:')
print(best_f1_row.round(4))

## Summary

- Logistic Regression wins on ROC-AUC (0.8445) — simpler and more interpretable than tree models for this dataset
- SMOTE on training set only avoids metric inflation from data leakage
- Top driver is `avg_monthly_spend` (engineered), validating the feature engineering work
- Threshold 0.5 is reasonable; slight recall gain possible at 0.35–0.40 if the business prefers catching more churners

Final model and logic formalised in `src/train.py` and `src/score.py`.